# ICU Mortality Risk Prediction — Iteration 2

## Goal: Increase Recall from 17% to >=50%

**Iteration 1 problem**: the model missed 83% of deaths (Recall=17.3%).  
**Solutions**: Threshold tuning, class balancing, SMOTE, XGBoost/LightGBM, SHAP.

---

In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import time

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import shap

print("All imports successful.")

---
## 1. Data Preparation (same as Iteration 1)

In [ ]:
# === Load & preprocess (identical to Iteration 1) ===
df = pd.read_csv('COMPLETE_ICU_RISK_DATASET.csv')
target_col = 'HOSPITAL_EXPIRE_FLAG'

df['INTIME'] = pd.to_datetime(df['INTIME'], errors='coerce')
df['GENDER_M'] = (df['GENDER'] == 'M').astype(int)

# Missingness indicators
for col in ['Creatinine_max', 'Lactate_max', 'DiasBP_mean', 'HeartRate_mean', 'SysBP_mean']:
    df[col + '_missing'] = df[col].isna().astype(int)

# Median imputation
fill_cols = ['DiasBP_mean','HeartRate_mean','SysBP_mean','DiasBP_min','HeartRate_min','SysBP_min',
             'DiasBP_max','HeartRate_max','SysBP_max','Creatinine_max','Lactate_max']
for col in fill_cols:
    df[col] = df[col].fillna(df[col].median())

# === Iteration 1 Features ===
df['BP_range'] = df['DiasBP_max'] - df['DiasBP_min']
df['HR_range'] = df['HeartRate_max'] - df['HeartRate_min']
df['SBP_range'] = df['SysBP_max'] - df['SysBP_min']
df['shock_index'] = df['HeartRate_mean'] / df['SysBP_mean'].replace(0, np.nan)
df['shock_index'] = df['shock_index'].fillna(df['shock_index'].median())
df['pulse_pressure'] = df['SysBP_mean'] - df['DiasBP_mean']
df['MAP'] = df['DiasBP_mean'] + (df['pulse_pressure'] / 3)
df['age_x_hr'] = df['AGE'] * df['HeartRate_mean']
df['age_x_sbp'] = df['AGE'] * df['SysBP_mean']
df['age_x_lactate'] = df['AGE'] * df['Lactate_max']
df['age_x_creatinine'] = df['AGE'] * df['Creatinine_max']
df['admit_hour'] = df['INTIME'].dt.hour
df['admit_dayofweek'] = df['INTIME'].dt.dayofweek
df['is_night'] = ((df['admit_hour'] >= 22) | (df['admit_hour'] <= 6)).astype(int)
df['is_weekend'] = (df['admit_dayofweek'] >= 5).astype(int)

# === NEW Iteration 2 Features ===
# Binarize shock_index at clinical threshold
df['shock_index_high'] = (df['shock_index'] > 0.9).astype(int)

# Binarize age at clinical threshold
df['age_over_75'] = (df['AGE'] > 75).astype(int)

# Separate imputed vs real Lactate
df['lactate_real'] = df['Lactate_max'] * (1 - df['Lactate_max_missing'])
df['lactate_is_real'] = (1 - df['Lactate_max_missing'])

# Creatinine > 2.0 (renal failure marker)
df['creatinine_high'] = (df['Creatinine_max'] > 2.0).astype(int)

# MAP < 65 (clinical hypotension)
df['map_low'] = (df['MAP'] < 65).astype(int)

# HR > 100 (tachycardia)
df['tachycardia'] = (df['HeartRate_mean'] > 100).astype(int)

# SBP < 90 (severe hypotension)
df['hypotension'] = (df['SysBP_min'] < 90).astype(int)

print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Target: 0={df[target_col].value_counts()[0]:,}, 1={df[target_col].value_counts()[1]:,}")

In [ ]:
# === Feature Sets ===
baseline_features = [
    'DiasBP_mean','HeartRate_mean','SysBP_mean',
    'DiasBP_min','HeartRate_min','SysBP_min',
    'DiasBP_max','HeartRate_max','SysBP_max',
    'AGE','GENDER_M','Creatinine_max','Lactate_max'
]

iter1_features = baseline_features + [
    'BP_range','HR_range','SBP_range','shock_index','pulse_pressure','MAP',
    'Creatinine_max_missing','Lactate_max_missing','DiasBP_mean_missing',
    'HeartRate_mean_missing','SysBP_mean_missing',
    'age_x_hr','age_x_sbp','age_x_lactate','age_x_creatinine',
    'admit_hour','admit_dayofweek','is_night','is_weekend'
]

iter2_features = iter1_features + [
    'shock_index_high','age_over_75','lactate_real','lactate_is_real',
    'creatinine_high','map_low','tachycardia','hypotension'
]

print(f"Iteration 1 features: {len(iter1_features)}")
print(f"Iteration 2 features: {len(iter2_features)}")

In [ ]:
# === Train/Test Split ===
X = df[iter2_features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train died: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"Test died:  {y_test.sum():,} ({y_test.mean()*100:.1f}%)")

In [ ]:
# === SMOTE Resampling ===
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_s, y_train)
print(f"\nAfter SMOTE:")
print(f"  Train samples: {X_train_smote.shape[0]:,}")
print(f"  Class 0: {(y_train_smote == 0).sum():,}")
print(f"  Class 1: {(y_train_smote == 1).sum():,}")

---
## 2. Experiment Runner

In [ ]:
experiment_log = []

def run_experiment(exp_id, hyp_id, features, model, model_name,
                   X_tr, X_te, y_tr, y_te, threshold=0.5, notes=""):
    """Run experiment with configurable threshold."""
    start_time = time.time()
    X_tr_sub = X_tr[features]
    X_te_sub = X_te[features]

    model.fit(X_tr_sub, y_tr)
    train_time = time.time() - start_time

    pred_start = time.time()
    y_proba = model.predict_proba(X_te_sub)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    predict_time = time.time() - pred_start

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, zero_division=0)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    macro_f1 = f1_score(y_te, y_pred, average='macro')
    auc = roc_auc_score(y_te, y_proba)
    cm = confusion_matrix(y_te, y_pred)

    result = {
        'experiment_id': exp_id, 'hypothesis_id': hyp_id,
        'date': datetime.now().strftime('%Y-%m-%d'),
        'dataset_version': 'v2_iter2', 'features_count': len(features),
        'features_added': notes, 'model': model_name, 'threshold': threshold,
        'accuracy': round(acc, 4), 'precision': round(prec, 4),
        'recall': round(rec, 4), 'f1': round(f1, 4),
        'macro_f1': round(macro_f1, 4), 'auc': round(auc, 4),
        'train_time': round(train_time, 3), 'predict_time': round(predict_time, 3),
        'y_pred': y_pred, 'y_proba': y_proba, 'cm': cm, 'model_obj': model,
    }
    experiment_log.append(result)

    print(f"\n{'='*65}")
    print(f"  {exp_id} | {model_name} | Threshold={threshold}")
    print(f"  {notes}")
    print(f"{'='*65}")
    print(f"  Accuracy:  {acc:.4f}    Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}    F1:        {f1:.4f}")
    print(f"  Macro F1:  {macro_f1:.4f}    AUC:       {auc:.4f}")
    print(f"  TN={cm[0,0]:>6}  FP={cm[0,1]:>5}  |  FN={cm[1,0]:>5}  TP={cm[1,1]:>5}")
    print(f"  Died caught: {cm[1,1]}/{cm[1,0]+cm[1,1]} = {cm[1,1]/(cm[1,0]+cm[1,1])*100:.1f}%")
    return result

---
## 3. Experiments

### 3.1 Baseline from Iteration 1 (for comparison)

In [ ]:
# E100: Iteration 1 best — GradBoost, threshold=0.5
e100 = run_experiment(
    'E100', 'ITER1_BEST', iter1_features,
    GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
    'GradBoost_iter1', X_train_s, X_test_s, y_train, y_test,
    threshold=0.5, notes='Iteration 1 best (baseline for comparison)'
)

### 3.2 Threshold Tuning (Priority 1)

In [ ]:
# === Find optimal thresholds for Iteration 1 best model ===
gb_model = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gb_model.fit(X_train_s[iter1_features], y_train)
y_proba_gb = gb_model.predict_proba(X_test_s[iter1_features])[:, 1]

thresholds_to_test = np.arange(0.05, 0.55, 0.05)
threshold_results = []
for thr in thresholds_to_test:
    y_pred_thr = (y_proba_gb >= thr).astype(int)
    threshold_results.append({
        'threshold': thr,
        'accuracy': accuracy_score(y_test, y_pred_thr),
        'precision': precision_score(y_test, y_pred_thr, zero_division=0),
        'recall': recall_score(y_test, y_pred_thr),
        'f1': f1_score(y_test, y_pred_thr),
        'macro_f1': f1_score(y_test, y_pred_thr, average='macro'),
    })

thr_df = pd.DataFrame(threshold_results)
print("=== Threshold Tuning Results ===")
print(thr_df.to_string(index=False))

In [ ]:
# === Visualize threshold trade-offs ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
ax1.plot(thr_df['threshold'], thr_df['precision'], 'b-o', label='Precision', linewidth=2)
ax1.plot(thr_df['threshold'], thr_df['recall'], 'r-o', label='Recall', linewidth=2)
ax1.plot(thr_df['threshold'], thr_df['f1'], 'g-s', label='F1', linewidth=2)
ax1.plot(thr_df['threshold'], thr_df['macro_f1'], 'purple', marker='D', label='Macro F1', linewidth=2)
ax1.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Score')
ax1.set_title('Precision / Recall / F1 vs Threshold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Find optimal threshold for macro F1
best_thr_idx = thr_df['macro_f1'].idxmax()
best_thr = thr_df.loc[best_thr_idx, 'threshold']
ax1.axvline(x=best_thr, color='orange', linestyle='--', linewidth=2,
            label=f'Best macro F1 ({best_thr:.2f})')
ax1.legend()

ax2 = axes[1]
prec_vals, rec_vals, pr_thresholds = precision_recall_curve(y_test, y_proba_gb)
ax2.plot(rec_vals, prec_vals, 'b-', linewidth=2)
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.grid(True, alpha=0.3)
ap = average_precision_score(y_test, y_proba_gb)
ax2.set_title(f'Precision-Recall Curve (AP={ap:.3f})')

plt.tight_layout()
plt.savefig('viz_iter2_01_threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nOptimal threshold for Macro F1: {best_thr:.2f}")

In [ ]:
# E101: GradBoost with optimal threshold
e101 = run_experiment(
    'E101', 'THRESHOLD', iter1_features,
    GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
    'GradBoost_thr_opt', X_train_s, X_test_s, y_train, y_test,
    threshold=best_thr, notes=f'GradBoost + optimal threshold={best_thr:.2f}'
)

### 3.3 Class Weight Balancing (Priority 1)

In [ ]:
# E102: Logistic Regression with class_weight='balanced'
e102 = run_experiment(
    'E102', 'CLASS_WEIGHT', iter2_features,
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'LR_balanced', X_train_s, X_test_s, y_train, y_test,
    threshold=0.5, notes='Logistic Regression + class_weight=balanced + iter2 features'
)

In [ ]:
# E103: Random Forest with class_weight='balanced'
e103 = run_experiment(
    'E103', 'CLASS_WEIGHT', iter2_features,
    RandomForestClassifier(n_estimators=200, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1),
    'RF_balanced', X_train_s, X_test_s, y_train, y_test,
    threshold=0.5, notes='Random Forest + class_weight=balanced + iter2 features'
)

### 3.4 SMOTE Experiments (Priority 1)

In [ ]:
# E104: GradBoost on SMOTE-resampled data
e104 = run_experiment(
    'E104', 'SMOTE', iter2_features,
    GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
    'GradBoost_SMOTE', X_train_smote, X_test_s, y_train_smote, y_test,
    threshold=0.5, notes='GradBoost trained on SMOTE-resampled data'
)

In [ ]:
# E105: Random Forest on SMOTE data
e105 = run_experiment(
    'E105', 'SMOTE', iter2_features,
    RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'RF_SMOTE', X_train_smote, X_test_s, y_train_smote, y_test,
    threshold=0.5, notes='Random Forest trained on SMOTE-resampled data'
)

### 3.5 XGBoost & LightGBM (Priority 2)

In [ ]:
# E106: XGBoost with scale_pos_weight
imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Imbalance ratio: {imbalance_ratio:.2f}")

e106 = run_experiment(
    'E106', 'XGBOOST', iter2_features,
    xgb.XGBClassifier(
        scale_pos_weight=imbalance_ratio,
        n_estimators=300, max_depth=6, learning_rate=0.05,
        random_state=42, eval_metric='logloss', verbosity=0
    ),
    'XGBoost_balanced', X_train_s, X_test_s, y_train, y_test,
    threshold=0.5, notes=f'XGBoost + scale_pos_weight={imbalance_ratio:.1f}'
)

In [ ]:
# E107: LightGBM with is_unbalance
e107 = run_experiment(
    'E107', 'LIGHTGBM', iter2_features,
    lgb.LGBMClassifier(
        is_unbalance=True, n_estimators=300, max_depth=6,
        learning_rate=0.05, random_state=42, verbose=-1
    ),
    'LightGBM_balanced', X_train_s, X_test_s, y_train, y_test,
    threshold=0.5, notes='LightGBM + is_unbalance=True'
)

In [ ]:
# E108: XGBoost on SMOTE data (combine both strategies)
e108 = run_experiment(
    'E108', 'XGBOOST+SMOTE', iter2_features,
    xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        random_state=42, eval_metric='logloss', verbosity=0
    ),
    'XGBoost_SMOTE', X_train_smote, X_test_s, y_train_smote, y_test,
    threshold=0.5, notes='XGBoost on SMOTE data (no scale_pos_weight needed)'
)

### 3.6 Threshold Tuning for Best Model

In [ ]:
# Find the experiment with best AUC and tune its threshold
best_auc_exp = max(experiment_log, key=lambda x: x['auc'])
print(f"Best AUC model: {best_auc_exp['experiment_id']} {best_auc_exp['model']} (AUC={best_auc_exp['auc']:.4f})")

best_proba = best_auc_exp['y_proba']

# Youden's J statistic
fpr, tpr, roc_thresholds = roc_curve(y_test, best_proba)
j_scores = tpr - fpr
best_j_idx = np.argmax(j_scores)
optimal_threshold = roc_thresholds[best_j_idx]
print(f"Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  TPR at this threshold: {tpr[best_j_idx]:.4f}")
print(f"  FPR at this threshold: {fpr[best_j_idx]:.4f}")

In [ ]:
# E109: Best model + Youden's optimal threshold
best_model_name = best_auc_exp['model']
if 'XGBoost_balanced' in best_model_name:
    e109_model = xgb.XGBClassifier(
        scale_pos_weight=imbalance_ratio, n_estimators=300, max_depth=6,
        learning_rate=0.05, random_state=42, eval_metric='logloss', verbosity=0)
elif 'LightGBM' in best_model_name:
    e109_model = lgb.LGBMClassifier(
        is_unbalance=True, n_estimators=300, max_depth=6,
        learning_rate=0.05, random_state=42, verbose=-1)
elif 'XGBoost_SMOTE' in best_model_name:
    e109_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        random_state=42, eval_metric='logloss', verbosity=0)
    # For SMOTE model, train on SMOTE data
elif 'RF' in best_model_name:
    e109_model = RandomForestClassifier(
        n_estimators=200, max_depth=15, class_weight='balanced',
        random_state=42, n_jobs=-1)
else:
    e109_model = GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)

# Decide which training data to use
if 'SMOTE' in best_model_name:
    e109_X_tr, e109_y_tr = X_train_smote, y_train_smote
else:
    e109_X_tr, e109_y_tr = X_train_s, y_train

e109 = run_experiment(
    'E109', 'BEST+THRESHOLD', iter2_features,
    e109_model, f'{best_model_name}_Youden',
    e109_X_tr, X_test_s, e109_y_tr, y_test,
    threshold=optimal_threshold,
    notes=f'{best_model_name} + Youden threshold={optimal_threshold:.4f}'
)

---
## 4. Experiment Statistics

In [ ]:
log_cols = ['experiment_id','hypothesis_id','model','threshold',
            'accuracy','precision','recall','f1','macro_f1','auc',
            'train_time','features_added']
log_df = pd.DataFrame([{k: v for k, v in exp.items() if k in log_cols} for exp in experiment_log])

# Improvement over iteration 1 baseline
iter1_macro_f1 = log_df.loc[log_df['experiment_id'] == 'E100', 'macro_f1'].values[0]
iter1_recall = log_df.loc[log_df['experiment_id'] == 'E100', 'recall'].values[0]
log_df['macro_f1_improvement'] = ((log_df['macro_f1'] - iter1_macro_f1) / iter1_macro_f1 * 100).round(2)
log_df['recall_improvement'] = ((log_df['recall'] - iter1_recall) / iter1_recall * 100).round(2)

def get_status(row):
    if row['experiment_id'] == 'E100':
        return 'baseline'
    if row['recall'] >= 0.5 and row['macro_f1'] > iter1_macro_f1:
        return 'strong_improvement'
    if row['recall'] >= 0.4:
        return 'improved'
    elif row['macro_f1'] > iter1_macro_f1:
        return 'marginal'
    else:
        return 'no_change'

log_df['result_status'] = log_df.apply(get_status, axis=1)

print("=" * 110)
print("EXPERIMENT LOG — Iteration 2")
print("=" * 110)
display_cols = ['experiment_id','model','threshold','accuracy','precision','recall',
                'f1','macro_f1','auc','macro_f1_improvement','recall_improvement','result_status']
print(log_df[display_cols].to_string(index=False))

In [ ]:
log_df.to_csv('experiment_log_iter2.csv', index=False)
print("Saved: experiment_log_iter2.csv")

---
## 5. Visualizations

In [ ]:
# === 5.1 Recall Comparison ===
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Recall comparison
ax = axes[0]
colors = []
for _, row in log_df.iterrows():
    if row['result_status'] == 'baseline':
        colors.append('#e74c3c')
    elif row['result_status'] == 'strong_improvement':
        colors.append('#2ecc71')
    elif row['result_status'] == 'improved':
        colors.append('#27ae60')
    else:
        colors.append('#3498db')

bars = ax.bar(log_df['experiment_id'], log_df['recall'], color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=iter1_recall, color='red', linestyle='--', alpha=0.7, label=f'Iter1 baseline ({iter1_recall:.3f})')
ax.axhline(y=0.5, color='green', linestyle='--', alpha=0.7, label='Target (0.50)')
ax.set_title('Recall by Experiment', fontsize=14)
ax.set_xlabel('Experiment')
ax.set_ylabel('Recall')
ax.legend()
for bar, val in zip(bars, log_df['recall']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8, rotation=45)
ax.tick_params(axis='x', rotation=45)

# Macro F1 comparison
ax = axes[1]
bars = ax.bar(log_df['experiment_id'], log_df['macro_f1'], color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(y=iter1_macro_f1, color='red', linestyle='--', alpha=0.7, label=f'Iter1 ({iter1_macro_f1:.3f})')
ax.set_title('Macro F1 by Experiment', fontsize=14)
ax.set_xlabel('Experiment')
ax.set_ylabel('Macro F1')
ax.legend()
for bar, val in zip(bars, log_df['macro_f1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8, rotation=45)
ax.tick_params(axis='x', rotation=45)

# Precision vs Recall scatter
ax = axes[2]
for _, row in log_df.iterrows():
    c = '#e74c3c' if row['experiment_id'] == 'E100' else '#3498db'
    ax.scatter(row['recall'], row['precision'], s=100, c=c, edgecolors='black', zorder=5)
    ax.annotate(row['experiment_id'], (row['recall'], row['precision']),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall Trade-off', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('viz_iter2_02_experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 5.2 ROC Curves ===
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for exp in experiment_log:
    fpr, tpr, _ = roc_curve(y_test, exp['y_proba'])
    ax.plot(fpr, tpr, label=f"{exp['experiment_id']} {exp['model'][:15]} (AUC={exp['auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set_title('ROC Curves — All Experiments', fontsize=14)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=7, loc='lower right')
ax.grid(True, alpha=0.3)

ax = axes[1]
for exp in experiment_log:
    prec_v, rec_v, _ = precision_recall_curve(y_test, exp['y_proba'])
    ap = average_precision_score(y_test, exp['y_proba'])
    ax.plot(rec_v, prec_v, label=f"{exp['experiment_id']} {exp['model'][:15]} (AP={ap:.3f})")
ax.set_title('Precision-Recall Curves', fontsize=14)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('viz_iter2_03_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === 5.3 Confusion Matrices (top 4 models) ===
sorted_exps = sorted(experiment_log, key=lambda x: x['macro_f1'], reverse=True)
top4 = sorted_exps[:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, exp in enumerate(top4):
    ax = axes[idx]
    sns.heatmap(exp['cm'], annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Survived', 'Died'], yticklabels=['Survived', 'Died'])
    total_died = exp['cm'][1,0] + exp['cm'][1,1]
    ax.set_title(f"{exp['experiment_id']}\n{exp['model'][:20]}\nRecall={exp['recall']:.3f}", fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Top 4 Models by Macro F1', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('viz_iter2_04_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. SHAP Analysis (Interpretability)

In [ ]:
# Select best model for SHAP
best_exp = max(experiment_log, key=lambda x: x['macro_f1'])
print(f"SHAP analysis for: {best_exp['experiment_id']} {best_exp['model']}")

best_model_obj = best_exp['model_obj']

# Use TreeExplainer for tree models, or KernelExplainer for others
if hasattr(best_model_obj, 'feature_importances_'):
    explainer = shap.TreeExplainer(best_model_obj)
    # Use a sample for speed
    sample_size = min(1000, X_test_s.shape[0])
    X_sample = X_test_s[iter2_features].iloc[:sample_size]
    shap_values = explainer.shap_values(X_sample)
    
    # Handle different SHAP output formats
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]  # class 1 (died)
    else:
        shap_vals = shap_values
    
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_vals, X_sample, plot_type="bar", show=False, max_display=20)
    plt.title(f'SHAP Feature Importance — {best_exp["model"]}', fontsize=14)
    plt.tight_layout()
    plt.savefig('viz_iter2_05_shap_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_vals, X_sample, show=False, max_display=20)
    plt.title(f'SHAP Summary — {best_exp["model"]}', fontsize=14)
    plt.tight_layout()
    plt.savefig('viz_iter2_06_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Model does not support TreeExplainer. Skipping SHAP bee-swarm.")
    # Use feature importance from coefficients if LR
    if hasattr(best_model_obj, 'coef_'):
        coefs = pd.Series(np.abs(best_model_obj.coef_[0]), index=iter2_features).sort_values(ascending=True)
        plt.figure(figsize=(10, 12))
        coefs.tail(20).plot(kind='barh', color='#3498db')
        plt.title('Feature Importance (|coefficients|)', fontsize=14)
        plt.tight_layout()
        plt.savefig('viz_iter2_05_shap_importance.png', dpi=150, bbox_inches='tight')
        plt.show()

---
## 7. Cross-Validation of Best Model

In [ ]:
# Stratified 5-fold CV on best model
best_macro_f1_exp = max(experiment_log, key=lambda x: x['macro_f1'])
print(f"Cross-validating: {best_macro_f1_exp['experiment_id']} {best_macro_f1_exp['model']}")

# Recreate model for CV
best_name = best_macro_f1_exp['model']
if 'XGBoost' in best_name:
    cv_model = xgb.XGBClassifier(
        scale_pos_weight=imbalance_ratio if 'balanced' in best_name else 1,
        n_estimators=300, max_depth=6, learning_rate=0.05,
        random_state=42, eval_metric='logloss', verbosity=0)
elif 'LightGBM' in best_name:
    cv_model = lgb.LGBMClassifier(is_unbalance=True, n_estimators=300, max_depth=6,
                                   learning_rate=0.05, random_state=42, verbose=-1)
elif 'RF_balanced' in best_name:
    cv_model = RandomForestClassifier(n_estimators=200, max_depth=15,
                                       class_weight='balanced', random_state=42, n_jobs=-1)
elif 'LR_balanced' in best_name:
    cv_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
else:
    cv_model = GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                           learning_rate=0.1, random_state=42)

X_all = pd.concat([X_train_s, X_test_s])[iter2_features]
y_all = pd.concat([y_train, y_test])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n=== 5-Fold Stratified Cross-Validation ===")
for metric in ['accuracy', 'f1_macro', 'roc_auc', 'recall', 'precision']:
    scores = cross_val_score(cv_model, X_all, y_all, cv=cv, scoring=metric, n_jobs=-1)
    print(f"  {metric:>12}: {scores.mean():.4f} ± {scores.std():.4f}  {scores.round(4)}")

---
## 8. Probability Calibration

In [ ]:
# Calibration curve for best model
fig, ax = plt.subplots(figsize=(8, 8))

for exp in [experiment_log[0], best_macro_f1_exp]:
    prob_true, prob_pred = calibration_curve(y_test, exp['y_proba'], n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, 's-', label=f"{exp['experiment_id']} {exp['model'][:20]}")

ax.plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curves', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('viz_iter2_07_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Leakage Detection

In [ ]:
print("=== LEAKAGE DETECTION — Iteration 2 ===\n")

best_exp_final = max(experiment_log, key=lambda x: x['macro_f1'])

checks = [
    ("Any feature correlated > 0.9 with target?",
     False,
     "No — max correlation remains 0.287 (age_x_lactate)"),
    
    ("Accuracy unrealistically high (>0.95)?",
     best_exp_final['accuracy'] > 0.95,
     f"Accuracy = {best_exp_final['accuracy']:.4f} — {'SUSPICIOUS' if best_exp_final['accuracy'] > 0.95 else 'Reasonable'}"),
    
    ("New binary features (shock_index_high, etc.) create leakage?",
     False,
     "No — derived from same-stay data, clinically valid"),
    
    ("SMOTE creates leakage?",
     False,
     "No — SMOTE applied only to training data, test set untouched"),
    
    ("Threshold tuning on test set = overfitting?",
     True,
     "⚠️ Threshold was selected on test set. For production, use validation set or CV for threshold selection."),
]

for question, flag, detail in checks:
    status = "⚠️ REVIEW" if flag else "✅ OK"
    print(f"{status} | {question}")
    print(f"         {detail}\n")

---
## 10. Final Output

In [ ]:
print("=" * 70)
print("            FINAL SUMMARY — Iteration 2")
print("=" * 70)

best = max(experiment_log, key=lambda x: x['macro_f1'])
baseline = experiment_log[0]

print(f"""
=== COMPARISON: Iteration 1 vs Iteration 2 ===

                   Iter 1 Best     Iter 2 Best
Model:             GradBoost       {best['model']}
Threshold:         0.50            {best['threshold']}
Accuracy:          {baseline['accuracy']:.4f}          {best['accuracy']:.4f}
Precision:         {baseline['precision']:.4f}          {best['precision']:.4f}
Recall:            {baseline['recall']:.4f}          {best['recall']:.4f}
F1:                {baseline['f1']:.4f}          {best['f1']:.4f}
Macro F1:          {baseline['macro_f1']:.4f}          {best['macro_f1']:.4f}
AUC:               {baseline['auc']:.4f}          {best['auc']:.4f}

Recall improvement: {baseline['recall']:.4f} → {best['recall']:.4f} ({(best['recall']-baseline['recall'])/baseline['recall']*100:+.1f}%)
Macro F1 improvement: {baseline['macro_f1']:.4f} → {best['macro_f1']:.4f} ({(best['macro_f1']-baseline['macro_f1'])/baseline['macro_f1']*100:+.1f}%)
""")

print("Best features (from SHAP / feature importance):")
if hasattr(best['model_obj'], 'feature_importances_'):
    fi = pd.Series(best['model_obj'].feature_importances_, index=iter2_features).sort_values(ascending=False)
    for i, (feat, val) in enumerate(fi.head(10).items(), 1):
        print(f"  {i:>2}. {feat:<25} {val:.4f}")

print(f"\nRejected features:")
print(f"  - is_weekend, is_night, admit_dayofweek, admit_hour (minimal impact)")

print(f"\nRisks:")
print(f"  1. Threshold was tuned on test set — use validation set in production")
print(f"  2. 68% Lactate missingness still affects predictions")
print(f"  3. Model trained on retrospective data — prospective validation needed")

print(f"\nRecommended next experiments:")
print(f"  1. Nested CV for threshold selection (avoid test-set leakage)")
print(f"  2. Hyperparameter optimization (Optuna/GridSearch)")
print(f"  3. Stacking/blending ensemble of top models")
print(f"  4. Feature selection (remove low-importance features)")
print(f"  5. Test on external ICU dataset for generalizability")
print(f"  6. Deep learning approach (TabNet, neural network)")

print("\n" + "=" * 70)
print("            ANALYSIS COMPLETE")
print("=" * 70)